# Insurance Risk Analytics - A/B Hypothesis Testing

## Objective
The objective of this stage is to statistically validate or reject key hypotheses about risk drivers within the portfolio. This phase provides the evidence base for a new, data-driven segmentation and pricing strategy.

This analysis aims to:
- Quantify risk using **Claim Frequency** and **Claim Severity** metrics.
- Test for significant risk differences across various **Provinces** and **Zip Codes**.
- Analyze **Margin (Profitability)** variances between different geographic locations.
- Investigate potential risk disparities between **Gender** groups to ensure fair and accurate pricing.
- Provide statistical evidence to support business recommendations for the underwriting team.

In [1]:
import pandas as pd
import sys
import os

# This tells Python to look in the parent directory for the 'src' folder
sys.path.append(os.path.abspath('..'))

from src.hypothesis_tests import calculate_metrics, perform_t_test, perform_chi_square_test

# Load data
df = pd.read_csv('../data/insurance_data.csv')
df = calculate_metrics(df)

print("Data loaded and KPIs calculated!")
df[['TotalClaims', 'HasClaim', 'Margin']].head()

Data loaded and KPIs calculated!


,TotalClaims,HasClaim,Margin
0,0.0,0,2346.0
1,9883.0,1,-7549.0
2,0.0,0,1697.0
3,12134.0,1,-9764.0
4,0.0,0,2582.0


In [2]:
df.columns

Index(['CustomerID', 'Age', 'Gender', 'Province', 'VehicleType',
       'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD',
       'PastClaims', 'Claimed', 'ClaimAmount', 'TotalPremium', 'TotalClaims',
       'CoverType', 'AutoMake', 'VehicleModel', 'CustomValueEstimate',
       'ZipCode', 'TransactionDate', 'HasClaim', 'Margin'],
      dtype='str')

# 1. Province Risk 
H₀: There are no risk differences across provinces.
KPI: Claim Severity (Numerical)
Group A (Control): Western Cape
Group B (Test): Gauteng

In [3]:
# Select Group A and Group B for Claim Severity
# Claim Severity is ONLY for policies that actually had a claim
gauteng_claims = df[(df['Province'] == 'Gauteng') & (df['HasClaim'] == True)]['TotalClaims']
western_cape_claims = df[(df['Province'] == 'Western Cape') & (df['HasClaim'] == True)]['TotalClaims']

t_stat, p_val = perform_t_test(gauteng_claims, western_cape_claims)

print(f"H0: No risk difference between Gauteng and Western Cape")
print(f"P-Value: {p_val:.5f}")

if p_val < 0.05:
    print("RESULT: Reject Null Hypothesis (Significant difference found)")
else:
    print("RESULT: Fail to Reject Null Hypothesis (No significant difference)")

H0: No risk difference between Gauteng and Western Cape
P-Value: 1.00000
RESULT: Fail to Reject Null Hypothesis (No significant difference)


# 2. Gender Risk 
H₀: There is no significant risk difference between Women and Men.
KPI: Claim Frequency (Categorical)
Group A: Men
Group B: Women

In [4]:
# We use Chi-Square because 'HasClaim' is True/False (Categorical)
# Filtering only for Men and Women (ignoring 'NotSpecified')
gender_df = df[df['Gender'].isin(['Male', 'Female'])]

chi2, p_val = perform_chi_square_test(gender_df, 'Gender', 'HasClaim')

print(f"Hypothesis 4 (Gender Risk - Frequency)")
print(f"P-Value: {p_val}")

if p_val < 0.05:
    print("Decision: REJECT Null Hypothesis")
else:
    print("Decision: FAIL TO REJECT")

Hypothesis 4 (Gender Risk - Frequency)
P-Value: 0.9638306173980254
Decision: FAIL TO REJECT


# 3. Zip Code Risk
H₀: There are no risk differences between zip codes.
KPI: Claim Frequency (Categorical - proportion of claims)
Group A: PostalCode 2000 (Johannesburg Central)
Group B: PostalCode 8000 (Cape Town Central)

In [8]:
# Automatically pick the top 2 most frequent ZipCodes to ensure we have data
top_zips = df['ZipCode'].value_counts().index[:2].tolist()
print(f"Testing ZipCodes: {top_zips}")

zip_risk_df = df[df['ZipCode'].isin(top_zips)]

# Running Chi-Square Test
chi2, p_val = perform_chi_square_test(zip_risk_df, 'ZipCode', 'HasClaim')

print(f"H0 (Zip Code Risk): P-Value = {p_val:.5f}")

if p_val < 0.05:
    print("Decision: REJECT Null Hypothesis")
else:
    print("Decision: FAIL TO REJECT")

Testing ZipCodes: [10004, 10002]
H0 (Zip Code Risk): P-Value = 0.22260
Decision: FAIL TO REJECT


# 4. Zip Code Margin (Profit)
H₀: There is no significant margin (profit) difference between zip codes.
KPI: Margin (Numerical: TotalPremium - TotalClaims)
Group A: PostalCode 2000
Group B: PostalCode 8000

In [10]:
# Using the same top 2 ZipCodes
zip_a = top_zips[0]
zip_b = top_zips[1]

margin_zip_a = df[df['ZipCode'] == zip_a]['Margin']
margin_zip_b = df[df['ZipCode'] == zip_b]['Margin']

# Running T-Test
t_stat, p_val = perform_t_test(margin_zip_a, margin_zip_b)

print(f"H0 (Zip Code Margin between {zip_a} and {zip_b}): P-Value = {p_val:.5f}")

if p_val < 0.05:
    print("Decision: REJECT Null Hypothesis")
else:
    print("Decision: FAIL TO REJECT")

H0 (Zip Code Margin between 10004 and 10002): P-Value = 0.26421
Decision: FAIL TO REJECT
